In [ ]:
# 파이토치 라이브러리 호출
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/rickiepark/hg-mldl/master/wine.csv'
df = pd.read_csv( url ) 

X = df[ ['alcohol', 'sugar', 'pH'] ] #
Y = df['class']           # (0:레드와인, 1:화이트와인)

from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y) 

from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
X_train_scaled = ss.fit_transform( X_train )  # (5197, 3) (5197,)
X_test_scaled = ss.transform( X_test )        # (1300, 3) (1300,)

# 딥러닝을 하기 위한 파이토치( 2의 제곱이 좋음. )
print(X_train_scaled.shape , Y_train.shape)
print(X_test_scaled.shape , Y_test.shape)
# torch.tensor ( 자료 , dtype=torch.타입비트수) : 텐서타입으로 변환
import torch
X_train = torch.tensor(X_train_scaled , dtype=torch.float32)
X_test = torch.tensor(X_test_scaled , dtype=torch.float32)

# 특성과 타겟의 차원수를 일치화
Y_train = torch.tensor(Y_train.values , dtype=torch.float32).unsqueeze(1)
Y_test = torch.tensor(Y_test.values , dtype=torch.float32).unsqueeze(1)

# 인공신경망 ANN 
import torch.nn as nn
# nn.Sequential( 입력층 -> 은닉층 -> 출력층 )
model = nn.Sequential(
    # 입력수
    # nn.Linear(피처수 , 파생피처수)
    nn.Linear(3,16),
    # 활성화함수
    # 음수값는 0으로 양수값은 유지
    nn.ReLU(),
    # 출력층
    # nn.Linear( 앞전 레이어의 피처수 , 결과 수 )
    nn.Linear(16,1)
)

# 손실함수
# 시그모이드 + cross Entropy 하나로 합친
import torch.optim as optim
optimizer = optim.Adam(model.parameters() , lr=0.01)
crierion = nn.BCEWithLogitsLoss()

# 에포크(학습수) 단위 학습(순전파 -> 역전파
for epoch in range( 10000 ) :
    model.train()
    # 이전 에포크의 계산된 기울기 초기화 
    optimizer.zero_grad()
    logits = model(X_train)
    # 손실(정답과 비교하여 손실 계산)
    loss = crierion(logits , Y_train)
    # (역전파) 5-5 각 가중치에 기울기 계산
    loss.backward()
    # (업데이트 반영) 가증치 반영
    optimizer.step()
    # ========== 확인용 ============
    if epoch % 100 == 0:
        print(f'에포크 : {epoch} , 손실(오차) : {loss.item()}')


# 모델 평가
# model.train() : 학습모드
# model.eval()  : 평가/추론모드
# 추론/평가 모드로 전환 
model.eval()
# 기울기 저장 안 함
with torch.no_grad() :
    # 이진분류모델
    # 0.5 보다 크면 1 , 아니면 0 
    train_pred = (torch.sigmoid(model(X_train)) >= 0.5).float()
    test_pred = (torch.sigmoid(model(X_test)) >= 0.5).float()  

# f1-score
from sklearn.metrics import f1_score
print(f1_score(Y_train.numpy() , train_pred , average='macro'))
print(f1_score(Y_test.numpy() , test_pred , average='macro'))
